# Notebook 12 — Logging for pipelines & services

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `11-collections-itertools-functools.ipynb`

**Next up:** `13-regex-text-extraction.ipynb`

---

## Learning objectives

- Configure module-level loggers instead of sprinkling `print`.
- Attach correlation-style fields without bespoke format strings everywhere.
- Understand propagation — avoid duplicate handlers during experiments.

## Table of contents

1. **Logger hierarchy**
2. **`LoggerAdapter` patterns**
3. **Propagation pitfall**
4. **Progressive drills — basic emit → retry banner → suppress noisy libraries**
5. **Exercise — timing helper logs duration**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Logger hierarchy

*Explanation:* **`logging.getLogger(__name__)`** preserves module paths in logs — grep-friendly once files multiply.


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")
log = logging.getLogger(__name__)

log.info("pipeline_start")
log.warning("high_latency_bucket", extra={"shard": "east", "p95_ms": 842})


## 2 · `LoggerAdapter` for correlation IDs

*Explanation:* **`LoggerAdapter`** prefixes repeated fields (`rid`, `tenant`) without rewriting every call site — JSON formatters plug in later.


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")


class RidAdapter(logging.LoggerAdapter):
    def process(self, msg, kwargs):
        rid = self.extra.get("rid", "-")
        return f"[rid={rid}] {msg}", kwargs


base = logging.getLogger("ingest.worker")
adapted = RidAdapter(base, {"rid": "trace-884"})
adapted.info("embedding_batch_started docs=%s tokens=%s", 128, 40_192)


## 3 · Propagation pitfall

*Explanation:* Child loggers bubble to parents — duplicate handlers cause duplicate lines; attach handlers at **one** level during notebooks.


In [ ]:
import logging

root = logging.getLogger()
root.handlers.clear()
logging.basicConfig(level=logging.INFO)

inner = logging.getLogger("outer.shard")
inner.info("single_line_only")


---

## Progressive drills — **easy → harder**

Production debugging rides on **consistent fields** — practice emitting retries, timeouts, and vendor IDs.


### A · Easiest — severity routing

Reserve WARNING for anomalies operators page on.


In [ ]:
import logging

logging.basicConfig(level=logging.DEBUG)
log = logging.getLogger("demo")

log.debug("tokenizer_cache_hit")
log.info("request_complete status=200")


### B · Medium — retry transcript

Structured retries explain flaky vendors faster than prose.


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("vendor")

for attempt in range(1, 4):
    sleep_s = round(0.2 * attempt, 3)
    log.warning("retry_schedule attempt=%s sleep_s=%s", attempt, sleep_s)


### C · Harder — silence chatty third-party logs

Flip libraries down to WARNING during demos.


In [ ]:
import logging

logging.basicConfig(level=logging.INFO)
logging.getLogger("urllib3").setLevel(logging.WARNING)

root = logging.getLogger()
root.info("your_signal_remains_visible")


### Exercise — log timed block

Implement `log_duration(log: logging.Logger, label: str)` as a **context manager** (using `contextlib.contextmanager`) that logs `START label` / `END label ms=...` at INFO using `time.perf_counter()`.


In [ ]:
import logging
from collections.abc import Iterator
from contextlib import contextmanager

logging.basicConfig(level=logging.INFO, format="%(message)s")


@contextmanager
def log_duration(log: logging.Logger, label: str) -> Iterator[None]:
    raise NotImplementedError


demo_log = logging.getLogger("timing")

with log_duration(demo_log, "fake_embed"):
    sum(range(1000))

print("done")


### Solution — log_duration

<details>
<summary>Click to expand</summary>

```python
import logging
import time
from collections.abc import Iterator
from contextlib import contextmanager

@contextmanager
def log_duration(log: logging.Logger, label: str) -> Iterator[None]:
    log.info("START %s", label)
    t0 = time.perf_counter()
    try:
        yield
    finally:
        ms = (time.perf_counter() - t0) * 1000
        log.info("END %s ms=%.2f", label, ms)
```

</details>
